# Cálculo de emissões por atividade produtiva

Esse notebook faz o cálculo das emissões por atividade produtiva.

## 1. Inicialização

Carrega dependências e executa configurações iniciais.

In [5]:
import numpy as np
import pandas as pd
import seaborn as sns

from redes import dados, emissoes, mip, modelo
from redes.visualizacoes import plotar_contabilidade_co2

sns.set_theme(context="notebook", style="whitegrid")

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 70)

## 2. Carregamento dos dados de entrada

### 2.1. MIP brasileira de 2015

A MIP é escolhida por um identificador do manifesto de entradas canônicas. O manifesto fixa o arquivo, sua fonte e seu SHA-256; a leitura rejeita qualquer conteúdo diferente.
Fonte: IBGE.

In [6]:
MIP = "mip_ibge_2015_67"

tabelas = dados.carregar_matriz_67(MIP)
mip.carregar_coeficientes_tecnicos_67(tabelas).head().iloc[:, :5]

Checksum validado: Matriz_de_Insumo_Produto_2015_Nivel_67.xls


atividade_destino,0191,0192,0280,0580,0680
atividade_origem,,,,,
0191,0.021056,0.027488,0.004699,0.000106,0.000032
0192,0.002413,0.032509,0.003979,0.000254,0.000086
0280,0.002699,0.007325,0.048358,0.000118,0.000007
0580,0.000514,0.002551,0.000358,0.015136,0.002867
0680,0.000020,0.000029,0.000010,0.000062,0.070423


### 2.2. Intensidades de CO₂ do Brasil

A escolha dos coeficientes é explícita pelos identificadores do manifesto. Cada arquivo é verificado antes da leitura. Os valores estão em Gg de CO₂ por R$ milhão de produção bruta.

Fonte: Sanguinet e Azzoni (2024).


In [4]:
COEFICIENTES_BRASIL = ["sanguinet_azzoni_2011", "sanguinet_azzoni_2018"]

coeficientes_2011 = dados.carregar_coeficientes_co2(COEFICIENTES_BRASIL[0])
coeficientes_2018 = dados.carregar_coeficientes_co2(COEFICIENTES_BRASIL[1])
matriz_coeficientes_co2 = pd.concat([coeficientes_2011, coeficientes_2018], axis=1)
matriz_coeficientes_co2.head()

SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2011.csv
SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2018.csv


ano,2011,2018
setor_id,,
S1,0.04,0.03
S2,0.05,0.04
S3,0.19,0.17
S4,0.07,0.08
S5,0.01,0.01


### 2.3. Parâmetros do exterior representativo

Declara os identificadores das intensidades e da inversa de Leontief do exterior representativo. O manifesto resolve cada identificador para um arquivo e SHA-256 exatos antes da leitura.

Neste cenário, assume-se que o exterior possui as mesmas intensidades de emissões e tecnologia que o Brasil.

Fonte: ver fontes em 2.1 e 2.2.


In [ ]:
COEFICIENTES_EXTERIOR = "exterior_proxy_brasil"
TECNOLOGIA_EXTERIOR = "exterior_proxy_brasil_2015"

intensidades_co2_exterior = dados.carregar_intensidades_co2_exterior(COEFICIENTES_EXTERIOR)
inversa_leontief_exterior = dados.carregar_inversa_leontief_exterior(TECNOLOGIA_EXTERIOR)

display(intensidades_co2_exterior.head())
display(inversa_leontief_exterior.head().iloc[:, :5])

## 3. Construir os coeficientes técnicos de Leontief

A Tabela 11 fornece `Bn`, os coeficientes de insumos nacionais (produto × atividade, 127 × 67). A Tabela 13 fornece `D`, a participação setorial da produção nacional (atividade × produto, 67 × 127). A composição `A = D @ Bn` produz a matriz de coeficientes técnicos intersetoriais (atividade × atividade). Em `A`, cada coluna `j` é dividida pela produção bruta da atividade `j`.

In [ ]:
matriz_bn = mip.carregar_matriz_bn_67(tabelas)
matriz_d = mip.carregar_matriz_participacao_67(tabelas)
coeficientes_tecnicos = modelo.calcular_coeficientes_tecnicos_67(tabelas)
print(f"Bn (produto × atividade): {matriz_bn.shape}")
print(f"D (atividade × produto): {matriz_d.shape}")
coeficientes_tecnicos

### Verificação da matriz A

A Tabela 14 (`D.Bn`) é a publicação do IBGE para a mesma composição.

In [ ]:
coeficientes_tecnicos_ibge = mip.carregar_coeficientes_tecnicos_67(tabelas)
erro_a = (coeficientes_tecnicos - coeficientes_tecnicos_ibge).abs().to_numpy().max()
print(f"Erro absoluto máximo em relação à Tabela 14 do IBGE: {erro_a:.2e}")
assert erro_a < 1e-10, "A matriz A calculada diverge da Tabela 14 do IBGE."

## 4. Inversa de Leontief

Para uma demanda final `f`, o modelo aberto é `(I − A)x = f`. Portanto, `x = Lf`, onde `L = (I − A)⁻¹` é a inversa de Leontief ou matriz de requerimentos totais.

In [ ]:
inversa_leontief = modelo.calcular_inversa_leontief(coeficientes_tecnicos)
inversa_leontief

In [ ]:
inversa_leontief_ibge = mip.carregar_inversa_leontief_ibge_67(tabelas)
erro_l = (inversa_leontief - inversa_leontief_ibge).abs().to_numpy().max()
print(f"Erro absoluto máximo em relação à Tabela 15 do IBGE: {erro_l:.2e}")
assert erro_l < 1e-10, "A inversa calculada diverge da matriz publicada pelo IBGE."

## 5. Coeficientes e inversa de Ghosh

O modelo de Ghosh usa os mesmos fluxos intersetoriais, mas normaliza as **linhas**: `B = diag(x)⁻¹ @ Z`. Aqui, `Z = D @ U` usa os usos intermediários nacionais `U` da Tabela 03, e `x` é a produção bruta por atividade obtida da Tabela 01. Cada linha de `B` informa como a produção da atividade fornecedora é alocada entre as atividades demandantes. A inversa de Ghosh é `G = (I − B)⁻¹`.

In [ ]:
matriz_transacoes = modelo.calcular_matriz_transacoes_intersetoriais_67(tabelas)
producao_bruta = modelo.calcular_producao_bruta_67(tabelas)
coeficientes_ghosh = modelo.calcular_coeficientes_alocacao_ghosh_67(tabelas)
print(f"Z (atividade × atividade): {matriz_transacoes.shape}")
print(f"Produção bruta: {producao_bruta.shape}")
coeficientes_ghosh

In [ ]:
inversa_ghosh = modelo.calcular_inversa_ghosh(coeficientes_ghosh)
identidade = pd.DataFrame(
    np.eye(67), index=coeficientes_ghosh.index, columns=coeficientes_ghosh.columns
)
residuo_ghosh = ((identidade - coeficientes_ghosh) @ inversa_ghosh - identidade).abs().to_numpy().max()
print(f"Resíduo máximo de (I − B) @ G − I: {residuo_ghosh:.2e}")
inversa_ghosh

## 6. Alinhamento das intensidades de CO₂

As intensidades carregadas na seção 2 são preparadas para os cálculos. Os setores `S1`–`S67` do artigo seguem a ordem das atividades da MIP. Essa correspondência por posição é a hipótese usada para atribuir os códigos brasileiros às intensidades. Em seguida, verificamos a ordem dos setores e a cobertura dos anos nos parâmetros externos.


In [ ]:
# Os setores S1-S67 do artigo seguem a ordem das atividades da MIP nivel 67.
intensidades_co2 = matriz_coeficientes_co2.copy()
intensidades_co2.index = producao_bruta.index
intensidades_co2.index.name = "atividade"

assert intensidades_co2_exterior.index.equals(producao_bruta.index)
assert inversa_leontief_exterior.index.equals(producao_bruta.index)
assert set(intensidades_co2.columns).issubset(intensidades_co2_exterior.columns)

intensidades_co2


## 7. Tres abordagens de contabilidade de CO2

**Producao territorial.** A emissao fica com a atividade que a gerou no Brasil: `e = gamma_BR * x_BR`. Ela inclui a producao destinada a exportacoes e exclui emissoes ocorridas no exterior.

**Consumo brasileiro.** A conta usa Leontief e exclui a demanda de exportacoes. Ela soma tres componentes: producao brasileira requerida pela demanda domestica, importacoes finais e importacoes intermediarias incorporadas na producao brasileira. Em notacao matricial: `C = gamma_BR_hat L_BR f_dom + gamma_EXT_hat L_EXT f_imp + gamma_EXT_hat L_EXT A_imp L_BR f_dom`.

**Renda no sistema domestico.** A conta usa Ghosh, `R = r_hat G_BR gamma_BR`, e atribui emissoes aos setores que recebem as entradas primarias `r`. Ela ainda nao separa renda brasileira e estrangeira; portanto, nao e uma reproducao internacional completa da responsabilidade baseada em renda de Marques et al. (2012).

Os parâmetros do exterior (`gamma_EXT` e `L_EXT`) são escolhidos pelos identificadores da seção 2.3 e resolvidos pelo manifesto em `raw/manifesto.csv`. O cenário inicial documenta `gamma_EXT = gamma_BR` e `L_EXT = L_BR`; ele pode ser substituído por um cenário externo empírico sem modificar as fórmulas. A conversão de produtos importados para atividades usa a matriz de participação nacional `D`, isto é, assume a mesma composição produto-atividade no exterior representativo.

In [ ]:
demanda_final = mip.carregar_demanda_final_nacional_67(tabelas)
demanda_final_domestica = mip.carregar_demanda_final_domestica_67(tabelas)
demanda_final_exportacoes = mip.carregar_demanda_final_exportacoes_67(tabelas)
demanda_final_importada = mip.carregar_demanda_final_importada_67(tabelas)
coeficientes_importados = modelo.calcular_coeficientes_importados_67(tabelas)
insumos_primarios = modelo.calcular_insumos_primarios_ghosh_67(tabelas)

erro_fechamento_demanda = (demanda_final - (demanda_final_domestica + demanda_final_exportacoes)).abs().max()
assert erro_fechamento_demanda < 1e-6
assert (insumos_primarios >= 0).all()
print(f"Demanda final domestica nacional: R$ {demanda_final_domestica.sum():,.0f} milhoes")
print(f"Exportacoes nacionais: R$ {demanda_final_exportacoes.sum():,.0f} milhoes")
print(f"Demanda final atendida por importacoes: R$ {demanda_final_importada.sum():,.0f} milhoes")
print(f"Entradas primarias do sistema de Ghosh: R$ {insumos_primarios.sum():,.0f} milhoes")

In [ ]:
resultados_por_ano = {}
totais_por_ano = {}
componentes_consumo_por_ano = {}

for ano, intensidade in intensidades_co2.items():
    emissoes_producao = emissoes.calcular_emissoes_producao(intensidade, producao_bruta)
    # Os parametros do exterior sao lidos de CSV. No cenario inicial, os
    # CSVs documentam a hipotese gamma_exterior = gamma_Brasil e L_exterior = L_Brasil.
    matrizes_consumo = emissoes.calcular_matrizes_emissoes_consumo_com_importacoes(
        intensidade, intensidades_co2_exterior[ano], inversa_leontief,
        inversa_leontief_exterior, demanda_final_domestica,
        demanda_final_importada, coeficientes_importados
    )
    matriz_exportacoes = emissoes.calcular_matriz_emissoes_consumo(
        intensidade, inversa_leontief, demanda_final_exportacoes
    )
    matriz_renda = emissoes.calcular_matriz_emissoes_renda(
        insumos_primarios, inversa_ghosh, intensidade
    )

    resultados_por_ano[ano] = pd.DataFrame({
        "producao": emissoes_producao,
        "consumo": matrizes_consumo["total"].sum(axis=0),
        "renda": matriz_renda.sum(axis=1),
    })
    totais_por_ano[ano] = resultados_por_ano[ano].sum()
    componentes_consumo_por_ano[ano] = pd.Series({
        "producao_domestica_para_consumo": matrizes_consumo["domestica"].to_numpy().sum(),
        "importacoes_finais": matrizes_consumo["importacoes_finais"].to_numpy().sum(),
        "importacoes_intermediarias": matrizes_consumo["importacoes_intermediarias"].to_numpy().sum(),
        "emissoes_brasileiras_para_exportacao": matriz_exportacoes.to_numpy().sum(),
    })

contabilidade_co2 = pd.concat(resultados_por_ano, names=["ano_intensidade", "atividade"])
totais_contabilidade_co2 = pd.DataFrame(totais_por_ano).T
totais_contabilidade_co2.index.name = "ano_intensidade"

componentes_consumo_co2 = pd.DataFrame(componentes_consumo_por_ano).T
componentes_consumo_co2.index.name = "ano_intensidade"

# A conta de consumo agora inclui CO2 externo estimado; por isso nao precisa coincidir
# com producao territorial ou renda calculada no sistema domestico.
display(totais_contabilidade_co2)
componentes_consumo_co2

In [ ]:
# Comparacao das 67 atividades para a intensidade de 2018; altere o ano se necessario.
descricoes_atividades = pd.Series(
    tabelas["14"].iloc[5:72, 1].to_numpy(),
    index=producao_bruta.index,
    name="descricao_atividade",
)

tabela_final_2018 = contabilidade_co2.xs(2018, level="ano_intensidade").join(descricoes_atividades)
tabela_final_2018 = tabela_final_2018[["descricao_atividade", "producao", "consumo", "renda"]]
tabela_final_2018.sort_values("producao", ascending=False)

### Gráfico, CSV e proveniência

As barras mostram consumo brasileiro com importações e renda no sistema doméstico; a linha mostra as emissões territoriais. A tabela e o gráfico são exportados em CSV e PNG em `outputs/`.

Cada leitura já valida o SHA-256 da fonte antes de calcular.

In [ ]:
pasta_outputs = dados.RAIZ_PROJETO / "outputs"
pasta_outputs.mkdir(exist_ok=True)
tabela_final_2018.to_csv(pasta_outputs / "contabilidade_co2_2018.csv", encoding="utf-8", lineterminator="\n")
figura_contabilidade, eixo_contabilidade = plotar_contabilidade_co2(
    tabela_final_2018, pasta_outputs / "contabilidade_co2_2018.png",
    titulo="Emissões de CO₂ por abordagem de contabilidade (intensidades de 2018)",
)
print(f"CSV e PNG salvos em: {pasta_outputs}")
figura_contabilidade

## 8. Inventario das tabelas da MIP

Cada chave de `tabelas` corresponde a uma aba do XLS e seu valor e um `pandas.DataFrame`.

In [ ]:
resumo_abas = pd.DataFrame(
    [(aba, *dataframe.shape) for aba, dataframe in tabelas.items()],
    columns=["aba", "linhas", "colunas"],
)
resumo_abas